# Notebook 05:
# Portfolio Construction — Volatility Targeting, Monthly Rebalancing, Walk-Forward Backtest, and Equity Curves

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px; margin-bottom: 15px;">
<strong>📘 PURPOSE:</strong> A reproducible portfolio construction and backtesting notebook that:

<div style="margin-left: 24px; margin-top: 8px;">

**(1)** loads the EWMA, unconstrained XGBoost, and DAG-constrained XGBoost risk forecasts produced by Notebooks 03 and 04 and aligns all three to the shared test-partition backtest window,

**(2)** implements a volatility-targeting allocation framework (10% annualized target) with inverse-volatility weighting, Ledoit–Wolf covariance shrinkage over a 63-day rolling window, no leverage (weights capped at [0, 1]), and cash residuals when forecasted volatility exceeds the target,

**(3)** executes monthly rebalancing (every 21 trading days) with next-day execution to preserve the label-safe governance logic established in Notebooks 03 and 04,

**(4)** applies a proportional transaction cost of 10 basis points per side (20 bps round-trip) at each rebalancing date,

**(5)** computes portfolio performance metrics — Sharpe ratio, maximum drawdown, Calmar ratio, annualized return, annualized volatility, and total transaction costs — for each of the three model-driven portfolio variants, and

**(6)** generates equity curves, drawdown comparisons, realized-versus-target volatility diagnostics, and weight time-series figures, then saves all artifacts to the locations specified by the capstone artifact contract.

</div>

Notebook 05 provides the first direct empirical test of <strong>Hypothesis H2 (Portfolio Risk Control)</strong> — the primary portfolio-level hypothesis of the capstone project.
</div>

---

## CAPSTONE CONTEXT

<div style="border-left: 4px solid #6a1b9a; padding-left: 12px; margin: 10px 0;">

**Title:**
<span style="color: purple;"><strong>Causal-Aware, Machine-Learning-Driven Risk Forecasting and Factor Construction:</strong></span> A Python–Azure Pipeline Integrating NLP, Directed Factor Constraints, and Portfolio Analytics

**Thesis:**
A small, theory-driven manually constrained <span style="color: purple;"><strong>Directed Acyclic Graph (DAG)</strong></span> that restricts information flow can improve the stability and interpretability of ML-based risk forecasting and factor-based portfolio allocation, relative to unconstrained baselines, under regime variation and estimation noise.

**Research Question:**
How does imposing <span style="color: purple;"><strong>manual causal constraints</strong></span> on an ML-driven risk forecasting pipeline affect forecast accuracy, portfolio performance, and interpretability compared to unconstrained baselines?

</div>

---

## MANUAL DAG (Conceptual Constraint Layer)

<div style="border: 2px solid #6a1b9a; background-color: #f3e5f5; padding: 12px; border-radius: 5px;">
<strong>🔗 DAG STRUCTURE:</strong>

```
Sentiment  →  Momentum  →  Returns
```
<span style="color: purple;">Path 1: Sentiment <strong>AMPLIFIES</strong> Momentum; Momentum <strong>FORECASTS</strong> Returns</span>

```
Value (HML)            →  Returns
```
<span style="color: purple;">Path 2: Value (HML) <strong>EXPLAINS</strong> Returns</span>

```
Volatility →  Risk     →  Allocation
```
<span style="color: purple;">Path 3: Volatility <strong>ESTIMATES</strong> Risk; Risk <strong>CONSTRAINS</strong> Allocation</span>

<span style="color: purple;"><strong>Notebook 05 Role in the DAG Pipeline:</strong></span> Notebooks 03 and 04 produced risk forecasts for 14 ETFs under three modeling regimes — EWMA (exponential smoothing), unconstrained XGBoost (all 151 non-sentiment features), and DAG-constrained XGBoost (74 VOL/MACRO/REGIME features only). Notebook 05 now feeds those forecasts into the <strong>Allocation node</strong> — the terminal node of the DAG — by translating each model's per-ETF volatility forecast into a portfolio weight via inverse-volatility targeting. The DAG-constrained model's allocation path flows strictly through Volatility → Risk → Allocation, with Momentum and Value excluded from the forecast signal that informs weights. The central empirical question of Notebook 05 is whether the DAG-constrained allocation exhibits lower maximum drawdown and improved risk-targeting stability during stress periods, compared to the unconstrained and EWMA-driven alternatives.
</div>

---

## HYPOTHESIS H2 — PORTFOLIO RISK CONTROL

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 HYPOTHESIS H2 — PORTFOLIO RISK CONTROL:</strong>

The DAG-constrained allocation exhibits lower maximum drawdown and improved risk-targeting stability during stress periods, compared to the unconstrained XGBoost-driven and EWMA-driven portfolio alternatives.

**Status prior to Notebook 05:** UNTESTED — Notebook 05 provides the first empirical test.

**Test mechanism:** Compare maximum drawdown, realized portfolio volatility versus the 10% annualized target, Sharpe ratio, and Calmar ratio across three portfolio variants constructed with identical volatility-targeting logic and transaction cost convention. The only source of variation across the three variants is the risk forecast signal driving the allocation: EWMA, unconstrained XGBoost, or DAG-constrained XGBoost.

**H2 confirmation criterion:** The CAUSAL_XGBOOST portfolio variant exhibits the smallest maximum drawdown and the smallest mean absolute deviation of realized portfolio volatility from the 10% target, across the full backtest window.

<span style="color: darkorange;"><strong>⚠ Scope note:</strong></span> Hypothesis H3 (NLP Sentiment) remains deferred — the SENT feature column is a 100% NaN placeholder in all three portfolio variants. Notebook 05 results are therefore identical to a Stage 1 ablation (no sentiment signal active).
</div>

---

## PORTFOLIO CONSTRUCTION APPROACH

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 DEFINITION — Inverse-Volatility Targeting with Portfolio Scaler:</strong>

At each monthly rebalancing date $t$, the allocation to each ETF $i$ is:

$$
w_i^{\text{raw}} = \frac{\sigma^{\text{target}}}{\hat{\sigma}_{i,t}} \times \frac{1}{N}
$$

where $\hat{\sigma}_{i,t}$ is the annualized volatility forecast for ETF $i$ produced at date $t$ by the active model (EWMA, XGBoost, or CAUSAL_XGBOOST), $\sigma^{\text{target}} = 0.10$ is the annualized portfolio volatility target, and $N = 14$ is the number of ETFs.

**Portfolio-level scaling (Ledoit–Wolf covariance check):**

After capping raw weights at $[0, 1]$, a second stage estimates the ex-ante portfolio volatility using a Ledoit–Wolf shrinkage covariance matrix estimated over the prior 63 trading days:

$$
\sigma^{\text{port}} = \sqrt{\mathbf{w}^{\top} \hat{\Sigma}_{\text{LW}} \, \mathbf{w}}
$$

When $\sigma^{\text{port}} > \sigma^{\text{target}}$, a scalar $\lambda = \min\!\left(1,\; \frac{\sigma^{\text{target}}}{\sigma^{\text{port}}}\right)$ is applied to all risky weights. Cash absorbs the residual:

$$
w_{\text{cash}} = \max\!\left(0,\; 1 - \sum_{i=1}^{N} w_i\right)
$$

This two-stage construction ensures that (a) the inverse-volatility logic distributes capital inversely to each ETF's own forecast risk, (b) the Ledoit–Wolf covariance check prevents ex-ante portfolio volatility from exceeding the target even when individual-asset forecasts are optimistic, and (c) no leverage is taken when aggregate forecasted risk is low.
</div>

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📘 DESIGN DECISIONS (FROZEN BEFORE NOTEBOOK 05):</strong>

| Decision | Frozen Value | Rationale |
|:---------|:-------------|:----------|
| **Volatility target** | 10% annualized | Commonly used institutional benchmark |
| **Allocation method** | Equal-risk-budget inverse-volatility targeting | Capstone focuses on risk control, not return maximization |
| **Leverage policy** | No leverage (weights capped at [0, 1]) | Cash absorbs excess when volatility exceeds target |
| **Transaction cost** | 10 bps per side (20 bps round-trip) | Applied to absolute weight change at each rebalancing date |
| **Rebalancing frequency** | Monthly — every 21 trading days | Matches the 20-day forecast horizon and walk-forward step from Notebooks 03–04 |
| **Backtest window** | Test partition from Notebooks 03–04 (2023-12-28 through 2025-12-31) | ~500 trading days, ~25 monthly rebalancing dates |
| **Covariance estimation** | Ledoit–Wolf shrinkage, 63-day rolling window | Ledoit & Wolf (2004); consistent with EWMA span and HML beta window |
| **Execution timing** | Next trading day after each decision date | Conservative assumption; avoids same-day look-ahead |
| **Forecast floor** | 1e-4 annualized volatility | Prevents division-by-zero in inverse-volatility computation |
</div>

---

## BACKTEST DESIGN AND TEMPORAL GOVERNANCE

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 DEFINITION — Walk-Forward Portfolio Backtest:</strong>

The portfolio backtest operates exclusively on the test partition — the final 20% of the effective modeling window — to ensure no training-period data influences the portfolio-level results. The backtest design follows a next-day execution convention:

| Stage | Timing | Description |
|:------|:-------|:------------|
| **Decision date** | End of day $t$ (rebalancing date) | Forecast produced by EWMA/XGBoost/CAUSAL_XGBOOST using data through $t$; Ledoit–Wolf covariance estimated from returns through $t$ |
| **Weight computation** | End of day $t$ | Inverse-volatility weights computed and scaled; cash residual determined |
| **Execution date** | Beginning of day $t+1$ | New portfolio weights take effect; transaction costs applied to absolute weight change |
| **Holding period** | $t+1$ through next execution date | Portfolio drifts with daily ETF returns; no intra-period rebalancing |
| **Return compounding** | Daily | Simple returns used for PnL; log returns used for volatility estimation |

<span style="color: darkblue;"><strong>Drift-aware weight update:</strong></span> Between rebalancing dates, portfolio weights drift as ETF prices move. The backtest engine tracks drifted weights day-by-day to correctly compute pre-trade versus post-trade weight changes at each subsequent rebalancing. This prevents the common backtest error of applying transaction costs to the full target weight rather than to the actual weight change.
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING — PORTFOLIO BACKTEST:</strong> Three leakage boundaries must hold simultaneously in Notebook 05.

**(1) Forecast leakage:** Volatility forecasts at each rebalancing date were produced in Notebooks 03 and 04 using only data available through that date. Notebook 05 trusts these artifact-sourced forecasts and performs no re-estimation.

**(2) Temporal leakage:** The Ledoit–Wolf covariance matrix at each rebalancing date $t$ is estimated exclusively from returns available through $t$. The 63-day lookback window uses <code>RETURNS_DF.loc[:decision_date].tail(63)</code> — strict boundary enforcement with no forward return information entering the covariance estimate.

**(3) Return leakage:** Daily portfolio returns during each holding segment use <code>SIMPLE_RETURNS_DF.loc[current_date]</code>, where <code>current_date</code> iterates forward from the execution date. No future segment returns are ever used to compute current-period PnL or drifted weights.

Log returns (not simple returns) are used for all volatility and covariance estimation. Simple returns are used for portfolio PnL compounding. The two return representations are maintained as separate DataFrames throughout Notebook 05.
</div>

---

## PERFORMANCE METRICS

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 DEFINITION — Portfolio Performance Summary Metrics:</strong>

**Annualized Return:**

$$
r_{\text{ann}} = \left(\prod_{t=1}^{T} (1 + r_t^{\text{net}})\right)^{\!252/T} - 1
$$

**Annualized Volatility:**

$$
\sigma_{\text{ann}} = \text{std}(\{r_t^{\text{log}}\}_{t=1}^{T},\; \text{ddof}=1) \times \sqrt{252}
$$

**Sharpe Ratio (annualized, excess over risk-free rate):**

$$
\text{Sharpe} = \frac{\overline{r^{\text{net}}} - \overline{r^{\text{rf}}}}{\text{std}(r^{\text{net}} - r^{\text{rf}},\; \text{ddof}=1)} \times \sqrt{252}
$$

**Maximum Drawdown:**

$$
\text{MaxDD} = \min_{t} \left(\frac{V_t}{\max_{s \leq t} V_s} - 1\right)
$$

**Calmar Ratio:**

$$
\text{Calmar} = \frac{r_{\text{ann}}}{|\text{MaxDD}|}
$$

**Risk-Targeting Stability:** Mean absolute deviation of the 21-day rolling realized portfolio volatility from the 10% annualized target, computed across all valid rolling windows in the backtest period.

<span style="color: darkblue;"><strong>Risk-free rate proxy:</strong></span> The daily risk-free rate is derived from the <code>MACRO__dtb3</code> column (3-month T-bill rate from FRED), annualized as <code>dtb3 / 100 / 252</code>. A fallback of zero is applied when the column is unavailable.
</div>

---

## INPUT AND OUTPUT FILES

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px;">
<strong>📁 INPUT (from Notebooks 01, 03, and 04):</strong>

- `data/processed/features.parquet` — Feature matrix (2,798 × 152); used to reconstruct the effective modeling mask and train/validation/test split boundaries
- `data/processed/target_fwd_vol.parquet` — Forward realized volatility target (2,798 × 14); used as reference for risk-targeting diagnostics
- `data/processed/etf_returns.parquet` — Daily log returns for 14 ETFs; used for Ledoit–Wolf covariance estimation and portfolio PnL compounding
- `reports/tables/notebook03_baseline_predictions_long.csv` — Long-form EWMA and XGBoost predictions (all 14 tickers, all test blocks)
- `reports/tables/notebook04_causal_predictions_long.csv` — Long-form DAG-constrained XGBoost predictions (all 14 tickers, all test blocks); fallback to `data/processed/notebook04_causal_predictions_long.csv` when the tables copy is absent
</div>

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📁 OUTPUT (produced by Notebook 05):</strong>

**Tables (`reports/tables/`):**
- `notebook05_portfolio_weights.csv` — Long-form rebalance weight table: one row per model × rebalance date × ticker; includes forecast volatility, pre-trade weight, raw weight, capped weight, final weight, cash weight, portfolio scaler, ex-ante volatility before and after scaling, turnover, and transaction cost
- `notebook05_portfolio_performance.csv` — Per-model performance summary: annualized return, annualized volatility, Sharpe ratio, maximum drawdown, Calmar ratio, final equity, and total transaction costs
- `notebook05_turnover_summary.csv` — Per-model turnover statistics: mean, median, and maximum monthly turnover; total transaction costs
- `notebook05_monthly_returns.csv` — Calendar-month compounded returns per model variant
- `notebook05_risk_targeting_stability.csv` — Risk-targeting stability metrics: mean and median 21-day realized volatility, RMSE versus 10% target, percentage of days within ±2% band, and mean ex-ante portfolio volatility
- `notebook05_artifact_manifest.csv` — Registry of all Notebook 05 output artifacts with file-existence validation flags

**Figures (`reports/figures/`):**
- `notebook05_equity_curves.png` — Cumulative growth of $1 for all three portfolio variants on a single axis
- `notebook05_drawdown_comparison.png` — Time-series drawdown overlay for all three portfolio variants
- `notebook05_realized_vs_target_volatility.png` — 21-day rolling realized portfolio volatility versus 10% target line for all three variants
- `notebook05_weight_timeseries.png` — Rebalancing-date weight step chart for the CAUSAL_XGBOOST portfolio (top-6 ETFs by average weight + cash)

**Data (`data/processed/`):**
- `notebook05_portfolio_returns_daily.csv` — Long-form daily net portfolio returns for all three variants; primary input for Notebook 06 ablation consumption
</div>

---

## NOTEBOOK 03 AND 04 RESULT CONTEXT (REFERENCE FOR NOTEBOOK 05 FRAMING)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 UPSTREAM MODEL PERFORMANCE SUMMARY (from Notebook 04 artifacts):</strong>

Notebook 05 translates the following forecast accuracy results into portfolio performance. All numbers below are artifact-verified per the Artifact-Truth Rule (Notebook 04 Handoff, Section 11.10).

| Model | Aggregate RMSE | Aggregate MAE | RMSE Wins vs XGBoost (of 14) | MAE Wins vs XGBoost (of 14) |
|:------|:--------------|:-------------|:-----------------------------|:----------------------------|
| EWMA (span-63) | 0.0708 | 0.0483 | — | — |
| XGBoost (unconstrained, 151 features) | 0.0698 | 0.0491 | — | — |
| CAUSAL_XGBOOST (DAG-constrained, 74 features) | 0.0811 | 0.0538 | 0 of 14 | 6 of 14 |

<span style="color: darkorange;"><strong>⚠ Hypothesis H1 status:</strong></span> H1 (forecast accuracy) was NOT CONFIRMED in Notebook 04. The DAG-constrained model exhibited an aggregate RMSE delta of +0.0113 (+16.2%) versus the unconstrained XGBoost baseline. The CAUSAL_XGBOOST model produced the observed ~0.76 volatility spike for SPY on 2025-04-10 — a period of elevated realized volatility consistent with the DAG-constrained model's more conservative feature set.

<span style="color: purple;"><strong>DAG hypothesis for Notebook 05:</strong></span> The DAG's economic intuition holds that a constraint-based allocation signal — even one that sacrifices some point forecast accuracy — may produce more stable portfolio risk exposure during stress periods, because the constraint excludes momentum cross-signals that can amplify drawdowns under correlated factor shocks. Notebook 05 tests whether the CAUSAL_XGBOOST portfolio translates a higher-RMSE forecast into a lower-drawdown, more volatility-stable allocation.
</div>

---

## FEATURE NAMING CONVENTION (REFERENCE)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 NAMING PATTERNS (inherited from Notebooks 02–04):</strong>

**Asset-level features** (one column per ETF per window):
<code>{DAG_NODE}__{TICKER}__{feature_name}__{window}</code>

**Prediction artifacts** (long-form, as loaded from Notebooks 03 and 04):
<code>date | ticker | model | y_pred | y_true</code>

**Examples of features from the DAG-constrained allowed set (Notebook 04 Risk stage):**
- `VOL__SPY__ewma_vol__span63` — Volatility node, SPY, EWMA volatility with span 63 (<strong>ALLOWED</strong> in DAG-constrained model)
- `MACRO__vixcls` — Macro conditioning, VIX closing level (<strong>ALLOWED</strong> in DAG-constrained model)
- `REGIME__vix_high` — Regime conditioning, binary high-VIX indicator (<strong>ALLOWED</strong> in DAG-constrained model)
- `MOM__SPY__cum_ret__21d` — Momentum node, SPY, 21-day cumulative return (<strong>EXCLUDED</strong> from DAG-constrained model)

The double-underscore delimiter convention from Notebook 02 carries through all downstream notebooks. Notebook 05 does not construct new features — Notebook 05 reads prediction artifacts directly.
</div>

---

## REPRODUCIBILITY

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ REPRODUCIBILITY GUARANTEES:</strong>

| Dimension | Policy |
|:----------|:-------|
| **Random Seed** | 692 (set in CODE_BLOCK_A; passed to NumPy and Python `random`) |
| **Time Ordering** | All backtest computations preserve strict temporal order; no future returns enter covariance or weight calculations |
| **Effective Modeling Mask** | Reuses the identical 60/20/20 completeness mask from Notebooks 03 and 04 to locate the test partition boundaries |
| **Forecast Source** | All volatility forecasts loaded from frozen Notebook 03 and 04 artifact CSVs — no model re-fitting in Notebook 05 |
| **Covariance Estimation** | Ledoit–Wolf shrinkage estimated exclusively from log returns available through the decision date; 63-day lookback window consistent with the EWMA span and HML beta window established in Notebook 02 |
| **Transaction Cost Convention** | 10 bps one-way applied to absolute weight change (risky assets only; cash incurs no cost) at each rebalancing date — applied on execution day (day 1 of each holding segment), not on decision day |
| **Drift Tracking** | Portfolio weights drift between rebalancing dates using actual daily ETF simple returns; the pre-trade weight at each subsequent rebalancing date reflects this drift, not a stale prior target weight |
| **Artifact Persistence** | All tables, figures, and daily return data saved to `reports/tables/`, `reports/figures/`, and `data/processed/` per the artifact contract |
| **Artifact-Truth Rule** | All performance numbers reported in the capstone report Section 4.3 must be read directly from `notebook05_portfolio_performance.csv` — no narrative approximation |
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING:</strong> Notebook 05 constructs and backtests portfolios. Three leakage boundaries must hold simultaneously: (1) <strong>Forecast leakage:</strong> volatility forecasts at each rebalancing date must have been produced in Notebooks 03 and 04 using only data available through that date — Notebook 05 performs no re-estimation. (2) <strong>Covariance leakage:</strong> the Ledoit–Wolf covariance matrix at decision date $t$ is estimated from <code>RETURNS_DF.loc[:t].tail(63)</code> — inclusive of $t$, exclusive of all future returns. (3) <strong>Return leakage:</strong> daily portfolio returns during each holding segment iterate forward from the execution date — no future segment returns ever enter current-period PnL or drifted weight calculations.
</div>

---

## NOTEBOOK STRUCTURE OVERVIEW

| Cell Block | Code Block | Purpose | Key Actions |
|:-----------|:-----------|:--------|:------------|
| **SETUP** | CODE_BLOCK_A | Environment initialization | Imports, paths, portfolio constants, display options, helper functions, output directory creation |
| **LOAD** | CODE_BLOCK_B | Load all inputs | Read features, target, returns Parquet files; load NB03 and NB04 prediction CSVs; validate index alignment; build TARGET_COL_MAP; extract risk-free rate proxy |
| **WINDOW** | CODE_BLOCK_C | Define backtest calendar | Recreate effective modeling mask and 60/20/20 split; compute date intersection across test window and all forecast artifacts; build rebalance schedule with next-day execution |
| **FORECASTS** | CODE_BLOCK_D | Build forecast matrices | Pivot long-form prediction CSVs to wide date-by-ticker matrices for EWMA, XGBOOST, and CAUSAL_XGBOOST; slice to common backtest calendar; build forecast coverage summary |
| **ALLOCATION** | CODE_BLOCK_E | Define and preview allocation logic | Implement Ledoit–Wolf covariance estimator; implement inverse-volatility targeting with portfolio scaler and cash residual; implement turnover function; preview first rebalancing date |
| **ENGINE** | CODE_BLOCK_F | Define portfolio backtest engine | Full walk-forward loop with drift-aware weights, rebalancing schedule, transaction costs; compute equity curve, running peak, drawdown, realized volatility; preview two-rebalancing test run |
| **RUN** | CODE_BLOCK_G | Execute full backtest for all three variants | Run `run_portfolio_backtest()` for EWMA, XGBOOST, and CAUSAL_XGBOOST; concatenate weight and daily return tables; save primary artifacts |
| **PERFORMANCE** | CODE_BLOCK_H | Compute performance summary table | Annualized return, annualized volatility, Sharpe ratio, maximum drawdown, Calmar ratio, final equity, total transaction costs; save `notebook05_portfolio_performance.csv` |
| **SECONDARY STATS** | CODE_BLOCK_I | Compute secondary summary tables | Calendar-month compounded returns; rebalancing-date turnover statistics; 21-day rolling realized volatility risk-targeting stability metrics; save three secondary CSVs |
| **FIGURES** | CODE_BLOCK_J | Generate all four report-ready figures | Equity curves; drawdown comparison; realized-versus-target volatility; CAUSAL_XGBOOST weight time-series; save four PNGs |
| **MANIFEST** | CODE_BLOCK_K | Artifact registry and final validation | Assemble artifact manifest with file-existence flags; print ALL_ARTIFACTS_EXIST confirmation; display final validation summary |

---

*Notebook 05 of 7 | MScFE 690 Capstone | Steven Archuleta | WorldQuant University*
